# Best-of-ablations CNN: wide StemEncoder + ELU + MaxPool + ModalMAE

This notebook combines the per-axis winners of the eight ablation notebooks in `Tutorials/Advanced/` (summarized in `reports/AdvancedComparisons_2026-09-29.html`) into one reconstructor, and checks whether the combination beats the individual winners:

| Axis | Choice | Source notebook |
|---|---|---|
| Architecture | `StemEncoderCNN` (grouped per-pupil stem + shared encoder) | `CNNArchitectureComparison.ipynb` |
| Activation | `ELU` everywhere (stem and encoder) | `ActivationComparison.ipynb` |
| Normalization | none | `NormalizationComparison.ipynb` |
| Downsampling | `MaxPool2d(2)` | `PoolingComparison.ipynb` |
| Loss | `ModalMAELoss` (`AI4AO/LossFunctions.py`) | `LossFunctionComparison.ipynb` |
| Size | ~3M parameters, obtained by **widening** (`base_channels` 7 → 12), never by adding layers | `CNNSizeComparison.ipynb`, `WidthDepthComparison.ipynb` |

Advanced architectures (ViT, ConvNeXt V2) are deliberately left out.

Everything else -- instrument (`CNNComparison_params.py`), `Trainer`, two-stage training recipe and budget, evaluation metric -- is identical to the ablation notebooks, so the result is directly comparable to their saved checkpoints, which are loaded as baselines at the end.

**Caveat:** each axis was ablated on its own, with one training run per arm (training-run noise ~5-10 nm). The winners are not guaranteed to combine additively -- that is what the baseline comparison below tests.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import ModalMAELoss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

Same shared `CNNComparison_params.py` as every ablation notebook (fictional nominal-geometry Pyramid WFS + 17-actuator DM, 190 modes, noise on). `wfs` and `dm` are built fresh and frozen with `.eval()`; only the reconstructor is trained.

The training budget is `TrainParams['TrainRunNb']` (10000) per stage, exactly as in the ablations. Note that `Trainer.train(training_steps, closed_loop_iterations)` counts *simulated closed-loop frames*: it takes `training_steps // closed_loop_iterations` optimizer steps, i.e. 10000 steps in stage 1 and 1000 in stage 2.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "BestCNN"
os.makedirs(PATH, exist_ok=True)

CLOSED_LOOP_ITERATIONS_STAGE2 = 10  # BPTT window in stage 2, matching the ablation notebooks

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()

loss = ModalMAELoss()

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")

## Architecture

`StemEncoderCNN` from `CNNArchitectureComparison.ipynb`, with two knobs factored out:

- `activation` -- the original hard-codes `nn.GELU`; here it is an argument so the same class serves both the new ELU model and the original GELU baseline (activations carry no parameters, so the baseline's saved `state_dict` loads into this class unchanged).
- `base_channels` -- the single width knob. It scales the grouped per-pupil stem (`4*b` then `8*b` channels) and, through it, every encoder stage (`8b, 16b, 32b`). Depth is untouched: the stem's one pooling plus `num_downsamples(Nout // 2)` VGG-style stages (3 at `Nout=42`), each two 3x3 convs + `MaxPool2d(2)`. No normalization layers anywhere.

In [ ]:
def num_downsamples(size, min_size=2):
    # How many stride-2 poolings a feature map of this size takes before dropping below min_size.
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation):
    # Two 3x3 convs, then a 2x MaxPool -- one VGG-style stage.
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


class StemEncoderCNN(nn.Module):
    # Grouped-conv stem processes each pupil image independently, then a shared
    # encoder mixes information across pupils; global-average-pool + linear head.

    def __init__(self, n_channels, Nmodes, Nout, base_channels=7, activation=nn.ELU):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(n_channels, base_channels * n_channels, kernel_size=11, padding=5, groups=n_channels),
            activation(),
            nn.Conv2d(base_channels * n_channels, 2 * base_channels * n_channels, kernel_size=7, padding=3, groups=n_channels),
            activation(),
            nn.MaxPool2d(2),
        )

        stem_out_channels = 2 * base_channels * n_channels
        n_stages = num_downsamples(Nout // 2)
        channels = [stem_out_channels] + [stem_out_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], activation)
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)

`base_channels=12` gives ~2.8M trainable parameters at this instrument's `Nout=42`/`Nmodes=190` (the closest to 3M; 13 gives ~3.3M). The sweep below makes that choice visible and re-checks it if `Nout`/`Nmodes` ever change.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for b in range(7, 15):
    print(f"base_channels={b:2d}: {count_params(StemEncoderCNN(n_channels, Nmodes, Nout, b)):>10,} parameters")

BASE_CHANNELS = 12
model = StemEncoderCNN(n_channels, Nmodes, Nout, base_channels=BASE_CHANNELS, activation=nn.ELU).to(device=device)
print(f"\nBestCNN (base_channels={BASE_CHANNELS}): {count_params(model):,} trainable parameters")

## Training

The same two-stage recipe as every ablation notebook, with a fresh `AdamW` per stage:

1. **Stage 1** -- `TrainParams['ClosedLoopIterations']` (a single pass) per optimizer step.
2. **Stage 2** -- `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per step, continuing from stage 1's weights.

Each stage resumes from `Data/BestCNN/BestCNN.pth` if it exists and saves back to it, so re-running the notebook continues training rather than restarting. Delete that file to train from scratch (e.g. after changing `BASE_CHANNELS`, which also triggers the shape-mismatch fallback below).

In [ ]:
CHECKPOINT = PATH / "BestCNN.pth"


def run_stage(closed_loop_iterations, label):
    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))
    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )
    # load_checkpoint already skips a missing file; RuntimeError is a shape mismatch against a
    # checkpoint saved with a different BASE_CHANNELS -- delete it to retrain from scratch.
    if os.path.exists(CHECKPOINT):
        trainer.load_checkpoint(CHECKPOINT, load_optimizer=False)
        print(f"{label}: resuming from {CHECKPOINT}")
    else:
        print(f"{label}: starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], closed_loop_iterations)
    trainer.save_checkpoint(CHECKPOINT)
    return trainer, loss_tracker, loss_tracker_ideal


trainer, loss_tracker_1, loss_tracker_ideal_1 = run_stage(TrainParams['ClosedLoopIterations'], "Stage 1 (single pass)")

In [ ]:
trainer, loss_tracker_2, loss_tracker_ideal_2 = run_stage(CLOSED_LOOP_ITERATIONS_STAGE2, "Stage 2 (closed loop)")

In [ ]:
# ModalMAE training loss vs. its oracle bound (ideal = loss computed from the true residual modes), per stage.
# The stages have different tracker lengths (one entry per optimizer step), so they are plotted separately.
trainer.plot_losses(loss_tracker_1, loss_tracker_ideal_1)
trainer.plot_losses(loss_tracker_2, loss_tracker_ideal_2, smoothing_window=20)

## Baselines

Saved checkpoints from the ablation notebooks, each the winner (or reference) of the axis the new model borrows from. All share this instrument and training budget; their architecture definitions are reproduced minimally here (`state_dict`-compatible with the originals).

| Baseline | Checkpoint | What it isolates |
|---|---|---|
| StemEncoder 0.9M (GELU) | `Data/CNNArchitectureComparison/StemEncoder.pth` | same architecture, before widening / ELU / ModalMAE |
| ClassicCNN 0.9M, ELU | `Data/ActivationComparison/ELU.pth` | activation winner alone |
| ClassicCNN 0.9M, ModalMAE | `Data/LossFunctionComparison/ModalMAE.pth` | loss winner alone |
| ClassicCNN 3M wide | `Data/WidthDepthComparison/3M_wide.pth` | same budget and width strategy, plain architecture |

The first two and the last were trained with `LogResidualVarianceLoss + Physics_loss`, the ablations' default; ModalMAE was trained with `ModalMAELoss`. Only the evaluation metric below is common to all.

In [ ]:
class ClassicCNN(nn.Module):
    # CNNArchitectureComparison.ipynb's ClassicCNN (plain VGG-style stack), activation as an argument.

    def __init__(self, n_channels, Nmodes, Nout, base_channels=28, activation=nn.LeakyReLU):
        super().__init__()
        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]
        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], activation)
        layers.append(nn.AdaptiveAvgPool2d(1))
        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        return self.head(self.encoder(x))


class PlainBlock(nn.Module):
    # WidthDepthComparison.ipynb's PlainBlock (named submodules, so its state_dict keys differ from ClassicCNN's).

    def __init__(self, in_channels, out_channels, activation=nn.LeakyReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act1 = activation()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act2 = activation()

    def forward(self, x):
        return self.act2(self.conv2(self.act1(self.conv1(x))))


class WideCNN(nn.Module):
    # WidthDepthComparison.ipynb's FlexCNN restricted to its "wide" case (blocks_per_stage=1, PlainBlock).

    def __init__(self, n_channels, Nmodes, Nout, base_channels):
        super().__init__()
        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]
        layers = []
        for i in range(n_stages):
            layers += [PlainBlock(channels[i], channels[i + 1]), nn.MaxPool2d(2)]
        layers.append(nn.AdaptiveAvgPool2d(1))
        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        return self.head(self.encoder(x))


baseline_specs = {
    "StemEncoder 0.9M (GELU)": (StemEncoderCNN(n_channels, Nmodes, Nout, base_channels=7, activation=nn.GELU),
                                DATA_DIR / "CNNArchitectureComparison" / "StemEncoder.pth"),
    "ClassicCNN 0.9M, ELU": (ClassicCNN(n_channels, Nmodes, Nout, activation=nn.ELU),
                             DATA_DIR / "ActivationComparison" / "ELU.pth"),
    "ClassicCNN 0.9M, ModalMAE": (ClassicCNN(n_channels, Nmodes, Nout),
                                  DATA_DIR / "LossFunctionComparison" / "ModalMAE.pth"),
    "ClassicCNN 3M wide": (WideCNN(n_channels, Nmodes, Nout, base_channels=51),
                           DATA_DIR / "WidthDepthComparison" / "3M_wide.pth"),
}

trainers = {f"BestCNN {count_params(model) / 1e6:.1f}M": trainer}
for name, (baseline, path) in baseline_specs.items():
    baseline = baseline.to(device=device)
    baseline_trainer = Trainer(
        wfs=wfs, framePreprocessor=framePreprocessor, dm=dm, M2C=M2C,
        phaseReconstructor=baseline, dataset=dataset, loss=loss,
        optimizer=torch.optim.AdamW(baseline.parameters()),
    )
    baseline_trainer.load_checkpoint(path, load_optimizer=False)  # strict: fails loudly if the architecture drifted
    trainers[name] = baseline_trainer
    print(f"{name:28s} {count_params(baseline):>10,} parameters  <- {path}")

## Closed-loop comparison (nm RMS)

Steady-state residual wavefront error over the pupil (first 30% of the rollout discarded as integrator warm-up, the same convention as `Trainer.evaluate()` and every ablation notebook), over `N_SEEDS` seeds shared by all models.

Unlike the ablation notebooks, both `torch` **and** `numpy` are reseeded before each rollout: `PhaseDataset` draws the number of turbulence layers from `np.random`, so reseeding `torch` alone does not give every model the same atmosphere. The paired Δ column is the per-seed difference to the new model (positive = baseline is worse), which removes the large seed-to-seed variation that the ± column still contains.

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    pupil_values = result.residual_opd[warmup:][..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


N_SEEDS = 1
seeds = [TrainParams['Seed'] + k for k in range(N_SEEDS)]

eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {name: [] for name in trainers}
for seed in seeds:
    for name, t in trainers.items():
        print(name)
        seed_everything(seed)
        result = t.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
        nm_rms[name].append(residual_nm_rms(result, dataset.pupil))
        del result

nm_rms = {name: np.array(v) for name, v in nm_rms.items()}
reference_name = next(iter(trainers))
reference = nm_rms[reference_name]

print(f"{'model':28s} {'nm RMS (mean ± sd)':>20s} {'paired Δ vs ' + reference_name:>26s}")
for name, v in nm_rms.items():
    delta = v - reference
    delta_str = "--" if name == reference_name else f"{delta.mean():+6.1f} ± {delta.std(ddof=1):4.1f}"
    print(f"{name:28s} {v.mean():12.1f} ± {v.std(ddof=1):5.1f} {delta_str:>26s}")

In [ ]:
names = list(nm_rms.keys())
means = [nm_rms[n].mean() for n in names]
sds = [nm_rms[n].std(ddof=1) for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#1c5cab"] + ["#86b6ef"] * (len(names) - 1)
ax.barh(names[::-1], means[::-1], xerr=sds[::-1], color=colors[::-1], capsize=3)
ax.set_xlabel(f"Steady-state residual (nm RMS), mean ± sd over {N_SEEDS} seeds")
ax.set_title("BestCNN vs. ablation winners")
plt.tight_layout()
plt.show()

## Closed loop of the new model

The same rollout animation as the ablation notebooks, for the new model.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 60
seed_everything(TrainParams['Seed'])
result = trainer.evaluate(n_steps=n_frames, dataset=eval_dataset)


def frame_panels(i):
    return [
        {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
        {"tensor": result.wfs_frames[i], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
    ]


fig, axes = imshow_multiple(frame_panels(0), max_channel_number=9)


def update(i):
    imshow_multiple(frame_panels(i), fig=fig, axes=axes, max_channel_number=9)
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())